# Lesson 5A: Principal Component Analysis - Theory

## Introduction

You are a data scientist working with gene expression data from cancer patients. Each patient has measurements for 20,000 genes. That is 20,000 features per patient! This creates several problems:

- **Computational cost**: Algorithms are slow with high dimensions
- **Visualization**: Impossible to plot 20,000-dimensional data
- **Curse of dimensionality**: Many algorithms fail in high dimensions
- **Interpretability**: Hard to understand which features matter

But here is the insight: **Most of the variation in your data might be captured by just a few combinations of these genes**. Maybe 99% of the variation can be explained by 50 principal components instead of 20,000 genes.

This is what **Principal Component Analysis (PCA)** solves. It is the most fundamental dimensionality reduction technique in machine learning.

**Real-world applications**:
- **Data compression**: Store images more efficiently
- **Visualization**: Plot high-dimensional data in 2D or 3D
- **Noise reduction**: Remove noise by keeping only important components
- **Feature engineering**: Create uncorrelated features for supervised learning
- **Anomaly detection**: Detect outliers in reconstruction error

In this lesson, we will:
1. Understand the curse of dimensionality and why we need dimensionality reduction
2. Derive PCA from the variance maximization principle
3. Understand eigenvectors, eigenvalues, and covariance matrices
4. Learn the SVD approach to PCA
5. Implement PCA from scratch using eigendecomposition
6. Determine how many components to keep using explained variance

Then in Lesson 5B, we will use scikit-learn for production PCA and apply it to real applications like face recognition (Eigenfaces).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits, load_iris
from sklearn.preprocessing import StandardScaler
from typing import Tuple
from numpy.typing import NDArray

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries loaded successfully!")

## The Curse of Dimensionality

As the number of dimensions increases, several problems emerge:

### 1. Data Becomes Sparse

In high dimensions, all points are far apart. Consider points uniformly distributed in a unit hypercube:

- 1D: Points fill a line of length 1
- 2D: Points fill a square of area 1
- 3D: Points fill a cube of volume 1
- nD: Points fill a hypercube of volume 1

But the volume of a sphere inscribed in the hypercube shrinks exponentially:

$$V_{\text{sphere}}(d) = \frac{\pi^{d/2}}{\Gamma(d/2 + 1)} \rightarrow 0 \text{ as } d \rightarrow \infty$$

**Implication**: In high dimensions, most data is near the edges, not in the center.

### 2. Distance Metrics Break Down

In high dimensions, all pairwise distances become similar:

$$\frac{d_{\text{max}} - d_{\text{min}}}{d_{\text{min}}} \rightarrow 0 \text{ as } d \rightarrow \infty$$

This means nearest neighbor methods stop working!

### 3. Exponential Sample Complexity

To maintain the same data density, the required number of samples grows exponentially with dimensions:

$$n_{\text{required}} \propto r^d$$

where $d$ is the number of dimensions.

**Example**: If you need 10 samples to cover a 1D space, you need $10^{10}$ samples to cover a 10D space with the same density!

In [ ]:
# Demonstrate curse of dimensionality
print("Demonstrating the curse of dimensionality...\n")

# Generate random points in different dimensions
n_points = 1000

for d in [2, 10, 50, 100]:
    # Generate random points in unit hypercube
    points = np.random.uniform(0, 1, (n_points, d))

    # Compute all pairwise distances
    from scipy.spatial.distance import pdist
    distances = pdist(points)

    # Compute statistics
    d_min = distances.min()
    d_max = distances.max()
    d_mean = distances.mean()
    d_std = distances.std()

    relative_contrast = (d_max - d_min) / d_min

    print(f"Dimension {d}:")
    print(f"  Min distance: {d_min:.4f}")
    print(f"  Max distance: {d_max:.4f}")
    print(f"  Mean distance: {d_mean:.4f}")
    print(f"  Std distance: {d_std:.4f}")
    print(f"  Relative contrast: {relative_contrast:.4f}")
    print()

print("💡 Notice how relative contrast shrinks as dimension increases!")
print("In high dimensions, all points are nearly equidistant!")

## PCA: The Variance Maximization View

PCA finds a low-dimensional representation that preserves as much variance (information) as possible.

### Problem Setup

Given:
- Data matrix $X \in \mathbb{R}^{m \times n}$ with $m$ samples and $n$ features
- Target dimensionality $k < n$

**Goal**: Find $k$ orthogonal directions (principal components) that capture the most variance.

### Derivation

**Step 1**: Center the data (subtract mean):
$$\bar{X} = X - \mu$$

where $\mu$ is the mean vector.

**Step 2**: Find first principal component $w_1$ that maximizes projected variance:

$$w_1 = \arg\max_{\|w\|=1} \text{Var}(Xw) = \arg\max_{\|w\|=1} w^T C w$$

where $C = \frac{1}{m}X^TX$ is the covariance matrix.

**Solution**: $w_1$ is the eigenvector of $C$ with largest eigenvalue:
$$Cw_1 = \lambda_1 w_1$$

**Step 3**: Find subsequent components $w_2, w_3, ...$ orthogonal to previous ones:

$$w_k = \arg\max_{\|w\|=1, w \perp w_1,...,w_{k-1}} w^T C w$$

**Solution**: These are eigenvectors with 2nd, 3rd, etc. largest eigenvalues.

### Key Insight

The principal components are the eigenvectors of the covariance matrix, ordered by their eigenvalues (which represent the variance captured).

## Covariance Matrix and Eigenvectors

### Covariance Matrix

For centered data $X \in \mathbb{R}^{m \times n}$, the covariance matrix is:

$$C = \frac{1}{m} X^T X \in \mathbb{R}^{n \times n}$$

Entry $C_{ij}$ measures how features $i$ and $j$ vary together:

$$C_{ij} = \frac{1}{m} \sum_{k=1}^{m} x_{ki} x_{kj}$$

**Properties**:
- Symmetric: $C^T = C$
- Positive semi-definite: $v^T C v \geq 0$ for all $v$
- Diagonal entries are variances: $C_{ii} = \text{Var}(X_i)$
- Off-diagonal entries are covariances: $C_{ij} = \text{Cov}(X_i, X_j)$

### Eigendecomposition

For symmetric matrix $C$:

$$C = Q \Lambda Q^T$$

where:
- $Q$ is orthogonal matrix of eigenvectors: $Q^T Q = I$
- $\Lambda$ is diagonal matrix of eigenvalues: $\Lambda = \text{diag}(\lambda_1, ..., \lambda_n)$

**Geometric interpretation**:
- Eigenvectors: Directions of principal axes
- Eigenvalues: Variance along each axis

### PCA Algorithm (Eigendecomposition)

1. **Center data**: $\bar{X} = X - \mu$
2. **Compute covariance**: $C = \frac{1}{m} \bar{X}^T \bar{X}$
3. **Eigendecomposition**: $C = Q \Lambda Q^T$
4. **Sort by eigenvalue**: Order eigenvectors by decreasing eigenvalue
5. **Select components**: Keep top $k$ eigenvectors as $W \in \mathbb{R}^{n \times k}$
6. **Project data**: $Z = \bar{X} W \in \mathbb{R}^{m \times k}$

In [ ]:
class PCA:
    """
    Principal Component Analysis implemented from scratch using eigendecomposition.

    PCA transforms data to a new coordinate system where the axes (principal components)
    are ordered by the amount of variance they explain.
    """

    def __init__(self, n_components: int = 2):
        """
        Initialize PCA.

        Args:
            n_components: Number of principal components to keep
        """
        self.n_components = n_components
        self.mean = None
        self.components = None  # Principal components (eigenvectors)
        self.explained_variance = None  # Eigenvalues
        self.explained_variance_ratio = None

    def fit(self, X: NDArray) -> 'PCA':
        """
        Fit PCA model by computing principal components.

        Args:
            X: Training data of shape (n_samples, n_features)

        Returns:
            self: Fitted estimator
        """
        n_samples, n_features = X.shape

        # Step 1: Center the data
        self.mean = np.mean(X, axis=0)
        X_centered = X - self.mean

        # Step 2: Compute covariance matrix
        # C = (1/n) X^T X
        cov_matrix = (X_centered.T @ X_centered) / n_samples

        # Step 3: Eigendecomposition of covariance matrix
        eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

        # Step 4: Sort eigenvectors by decreasing eigenvalue
        idx = eigenvalues.argsort()[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        # Step 5: Select top k components
        self.components = eigenvectors[:, :self.n_components].T  # Shape: (n_components, n_features)
        self.explained_variance = eigenvalues[:self.n_components].real

        # Compute explained variance ratio
        total_variance = eigenvalues.sum().real
        self.explained_variance_ratio = self.explained_variance / total_variance

        return self

    def transform(self, X: NDArray) -> NDArray:
        """
        Project data onto principal components.

        Args:
            X: Data of shape (n_samples, n_features)

        Returns:
            X_transformed: Projected data of shape (n_samples, n_components)
        """
        # Center data using training mean
        X_centered = X - self.mean

        # Project onto principal components
        # Z = X_centered @ W^T where W are the components
        X_transformed = X_centered @ self.components.T

        return X_transformed

    def fit_transform(self, X: NDArray) -> NDArray:
        """
        Fit PCA and transform data in one step.

        Args:
            X: Training data of shape (n_samples, n_features)

        Returns:
            X_transformed: Projected data of shape (n_samples, n_components)
        """
        self.fit(X)
        return self.transform(X)

    def inverse_transform(self, X_transformed: NDArray) -> NDArray:
        """
        Reconstruct original data from principal components.

        Args:
            X_transformed: Projected data of shape (n_samples, n_components)

        Returns:
            X_reconstructed: Reconstructed data of shape (n_samples, n_features)
        """
        # Z @ W + mean where W are the components
        X_reconstructed = X_transformed @ self.components + self.mean
        return X_reconstructed

print("✅ PCA class implemented successfully!")

In [ ]:
# Test PCA on Iris dataset
print("Loading Iris dataset...")
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names = iris.feature_names

print(f"Original data shape: {X_iris.shape}")
print(f"Features: {feature_names}\n")

# Fit PCA
print("Fitting PCA to reduce from 4D to 2D...")
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_iris)

print(f"\n✅ PCA complete!")
print(f"Transformed data shape: {X_pca.shape}")
print(f"\nExplained variance by component:")
for i, (var, ratio) in enumerate(zip(pca.explained_variance, pca.explained_variance_ratio)):
    print(f"  PC{i+1}: {var:.4f} ({ratio*100:.2f}%)")
print(f"\nTotal variance explained: {pca.explained_variance_ratio.sum()*100:.2f}%")

# Visualize
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_iris,
                      cmap='viridis', alpha=0.6, edgecolors='k', s=50)
plt.xlabel(f'First Principal Component ({pca.explained_variance_ratio[0]*100:.1f}% variance)',
           fontsize=12)
plt.ylabel(f'Second Principal Component ({pca.explained_variance_ratio[1]*100:.1f}% variance)',
           fontsize=12)
plt.title('Iris Dataset Projected onto First 2 Principal Components',
          fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Species')
plt.grid(True, alpha=0.3)
plt.show()

print("\n💡 The 3 species are well-separated in the 2D PCA space!")
print("We reduced from 4D to 2D while keeping {:.1f}% of the variance!".format(
    pca.explained_variance_ratio.sum()*100))